# 22 — 2단계 팔 하나를 **병변 보존 창**으로 다시 배워 갈아 끼우기 (STEP 53, 캐글 T4)

## 묻는 것 하나

STEP 49 파일럿(작은 모델 · 1만 장)에서 *"학습 때 창을 흔들면 위치 강건성 +25%p, clean −0.02"* 를 봤습니다.
그게 **배포 팔 하나를 실제로 갈아 끼웠을 때 보호자 조건 커버리지**(작업 규칙 7)로 이어지는지 봅니다.

- 대조군: 지금 배포 `stage2_effnetv2_s_m2.5_384_moderate` — **재학습 없이 배포 가중치 그대로** (예산 절약, 노트북 09 가 배운 팔)
- 처치: **같은 데이터 · 같은 레시피**(effnetv2_s · 384 · `moderate` · `default` 증강 · 10 epoch · fold 0) 에
  학습 크롭만 `m2.5` 크롭 안에서 **병변(중앙 40%)을 보존하는 창 (변 0.42~1.0, 위치 무작위)** 으로 흔듦 (`CFG.train_window_side`)
- 검증 변환은 그대로 (창 안 흔듦) → 이 노트북의 val macro-F1 은 **라벨 네모 조건** 값. 채택 판정은 로컬 프록시에서.

## 붙일 것 (Add Input)

| 입력 | 필수 |
|---|---|
| `dogskin-m25-step16` (m2.5 크롭) | ✅ |
| `dogskin-manifest-365k` | ✅ |
| `release` (STEP 16 릴리스 — 설정 참조) | ✅ |

**Accelerator: GPU T4** · Internet ON → **Save & Run All (Commit)**. holdout 은 안 엽니다.
사전등록: [`STEP53`](../docs/results/STEP53_2단계_병변보존창_팔교체_사전등록.md) — 관문은 여기서 안 바꿉니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "main"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.5"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 1. 데이터와 시간 — 노트북 09 와 같은 분할 (fold 0 · holdout 제외)

In [ ]:
import sys
sys.path.insert(0, DIR)
import torch
from src import crop, env, labels, split, stages, experiments
from src.config import CFG

env.load_prepared()
env.require_gpu()
DEV = "cuda"

# ── 여기만 바꾸면 됩니다 ───────────────────────────────────
TAG    = "m2.5"                # 배포 팔과 같은 크롭
MODEL  = "effnetv2_s"          # 배포 팔과 같은 백본 (노트북 09 · STEP 23)
EPOCHS = 10                    # 노트북 09 와 같음 (best epoch 4 였음, 조기 종료 patience 5)
SUBSET = 1.0
WINDOW = (0.42, 1.0)           # 창 변 / 크롭 변. 0.42 = 중앙 40% 병변 상자 + 여유. 상한 1.0 = 크롭 전체
# ──────────────────────────────────────────────────────────

mpath = env.work_root() / "manifests" / "manifest_final.parquet"
df = labels.load(mpath)
print(f"{len(df):,}행")
if len(df) < 300_000:
    print("[!] 365,428 보다 훨씬 적습니다 — 옛 데이터일 수 있습니다.")

have = crop.available_tags()
print("붙어 있는 태그:", have)
if TAG not in have:
    raise SystemExit(f"[X] 태그가 없습니다: {TAG}. dogskin-m25-step16 을 Add Input 하세요.")

keep = crop.chunks_with_crops(df, [TAG])
if not keep:
    raise SystemExit("[X] m2.5 크롭이 있는 청크가 없습니다.")
df = df[df["chunk"].isin(keep)].reset_index(drop=True)
print(f"\n쓸 청크 {keep} — {len(df):,}행")

view = stages.to_stage2(crop.switch_tag(df, TAG, verbose=False))
split.verify(view, fold=0, strict=True)
tr, va = split.get_fold(view, CFG().use_fold)
print(f"2단계 train {len(tr):,} / val {len(va):,}  (holdout 제외)")
print("클래스별 val:", va["label"].value_counts().sort_index().to_dict())

est = experiments.estimate_runtime([MODEL], img_size=384, n_train=int(len(tr) * SUBSET),
                                   epochs=EPOCHS, device=DEV)
print("\n[!] 추정치입니다 — 실측이 아닙니다. 노트북 09 실측: 이 팔 하나에 약 4.5h (T4).")

## 2. 처치 팔 하나만 배웁니다 (대조군 = 배포 가중치 그대로)

In [ ]:
import json, time
from pathlib import Path
import numpy as np
from src import train
from src.config import CLASSES

t0 = time.time()
r = experiments.train_and_measure(
    view, stage=2, img_size=384, crop_tag=TAG, device=DEV,
    model_name=MODEL, finetune="moderate", aug="default",
    epochs=EPOCHS, subset_frac=SUBSET,
    train_window=WINDOW,                       # ★ 처치 — 이것 하나만 배포 팔과 다릅니다
    measure_robust=True, measure_blur=False, n_robust=2000)

exp = r["exp_name"]
print(f"\n실험 이름 {exp}")
assert exp.endswith("_safe0.42"), f"이름에 처치가 안 붙었습니다: {exp}"
print(f"  macro-F1 {r['macro_f1']:.4f} · A6 recall {r['a6_recall']:.3f} · 배율 하락 {r.get('scale_drop', float('nan')):.1%}"
      f" · best epoch {r['best_epoch']} / {r['n_epochs']} · {(time.time()-t0)/60:.0f}분")
print("  ⚠️ 이 값은 라벨 네모(val) 조건입니다 — 배포 팔 val macro-F1 0.5894(STEP 23) 옆에 참고만. 채택 판정은 로컬 프록시.")

## 3. 산출물 — best.pt 를 릴리스 폴더 규격(`checkpoints/<실험>/`)으로 담아 ZIP

In [ ]:
import shutil, zipfile
OUTDIR = Path("/kaggle/working/step53_safe_arm")
shutil.rmtree(OUTDIR, ignore_errors=True)
src_dir = train.ckpt_dir(exp)
dst = OUTDIR / "checkpoints" / exp
dst.mkdir(parents=True)
for name in ["best.pt", "result.json", "history.csv", "config.json", "logits_val.npz"]:
    p = src_dir / name
    if p.exists():
        shutil.copy2(p, dst / name)
print("담은 파일:", sorted(q.name for q in dst.iterdir()))
assert (dst / "best.pt").exists(), "best.pt 가 없습니다"

summary = {"step": "STEP 53 — 2단계 병변 보존 창, 배포 팔 교체 확대 실험", "exp_name": exp, "window": list(WINDOW),
           "model": MODEL, "tag": TAG, "epochs": EPOCHS, "subset_frac": SUBSET, "chunks": keep,
           "n_train": r["n_train"], "run": {k: v for k, v in r.items() if k != "report"}}
(OUTDIR / "step53_kaggle.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
zpath = Path("/kaggle/working/step53_safe_arm.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as zf:
    for p in OUTDIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUTDIR))
print(f"저장: {zpath}  ({zpath.stat().st_size/1e6:.0f} MB) ← Output 탭에서 받아 로컬 판정에 씁니다")

## 다음 — 판정은 로컬에서 (보호자 조건 · 배포 3팔)

```
# 릴리스 사본을 만들고 effnetv2_s m2.5 팔만 safe 팔로 바꿉니다
cp -r data/work/hf_release data/work/hf_release_step53
rm -rf data/work/hf_release_step53/checkpoints/stage2_effnetv2_s_m2.5_384_moderate
unzip step53_safe_arm.zip -d data/work/step53 && cp -r data/work/step53/checkpoints/* data/work/hf_release_step53/checkpoints/
uv run --extra train python tools/detect_coverage.py --n 2500 --release data/work/hf_release_step53 \
    --detector data/work/detect_step50/detect_best.pt --out reports/detect_coverage_step53_safe_3arm.json
```

관문(사전등록): `user` 커버리지 − 31.6% ≥ +5%p · `label` 커버리지 ≥ 58.8% − 2%p. 어느 쪽이든 `docs/results/STEP53_*.md` 에 남깁니다.